# From Fully Connected Layers to Convolutions
:label:`sec_why-conv`

To this day,
the models that we have discussed so far
remain appropriate options
when we are dealing with tabular data.
By tabular, we mean that the data consist
of rows corresponding to examples
and columns corresponding to features.
With tabular data, we might anticipate
that the patterns we seek could involve
interactions among the features,
but we do not assume any structure *a priori*
concerning how the features interact.

Sometimes, we truly lack the knowledge to be able to guide the construction of fancier architectures.
In these cases, an MLP
may be the best that we can do.
However, for high-dimensional perceptual data,
such structureless networks can grow unwieldy.

For instance, let's return to our running example
of distinguishing cats from dogs.
Say that we do a thorough job in data collection,
collecting an annotated dataset of one-megapixel photographs.
This means that each input to the network has one million dimensions.
Even an aggressive reduction to one thousand hidden dimensions
would require a fully connected layer
characterized by $10^6 \times 10^3 = 10^9$ parameters.
Unless we have lots of GPUs, a talent
for distributed optimization,
and an extraordinary amount of patience,
learning the parameters of this network
may turn out to be infeasible.

A careful reader might object to this argument
on the basis that one megapixel resolution may not be necessary.
However, while we might be able
to get away with one hundred thousand pixels,
our hidden layer of size 1000 grossly underestimates
the number of hidden units that it takes
to learn good representations of images,
so a practical system will still require billions of parameters.
Moreover, learning a classifier by fitting so many parameters
might require collecting an enormous dataset.
And yet today both humans and computers are able
to distinguish cats from dogs quite well,
seemingly contradicting these intuitions.
That is because images exhibit rich structure
that can be exploited by humans
and machine learning models alike.
Convolutional neural networks (CNNs) are one creative way
that machine learning has embraced for exploiting
some of the known structure in natural images.


## Invariance

Imagine that we want to detect an object in an image.
It seems reasonable that whatever method
we use to recognize objects should not be overly concerned
with the precise location of the object in the image.
Ideally, our system should exploit this knowledge.
Pigs usually do not fly and planes usually do not swim.
Nonetheless, we should still recognize
a pig were one to appear at the top of the image.
We can draw some inspiration here
from the children's game "Where's Waldo"
(which itself has inspired many real-life imitations, such as that depicted in :numref:`img_waldo`).
The game consists of a number of chaotic scenes
bursting with activities.
Waldo shows up somewhere in each,
typically lurking in some unlikely location.
The reader's goal is to locate him.
Despite his characteristic outfit,
this can be surprisingly difficult,
due to the large number of distractions.
However, *what Waldo looks like*
does not depend upon *where Waldo is located*.
We could sweep the image with a Waldo detector
that could assign a score to each patch,
indicating the likelihood that the patch contains Waldo. 
In fact, many object detection and segmentation algorithms 
are based on this approach :cite:`Long.Shelhamer.Darrell.2015`. 
CNNs systematize this idea of *spatial invariance*,
exploiting it to learn useful representations
with fewer parameters.

![Can you find Waldo (image courtesy of William Murphy (Infomatique))?](../img/waldo-football.jpg)
:width:`400px`
:label:`img_waldo`

We can now make these intuitions more concrete 
by enumerating a few desiderata to guide our design
of a neural network architecture suitable for computer vision:

1. In the earliest layers, our network
   should respond similarly to the same patch,
   regardless of where it appears in the image. This principle is called *translation invariance* (or *translation equivariance*).
1. The earliest layers of the network should focus on local regions,
   without regard for the contents of the image in distant regions. This is the *locality* principle.
   Eventually, these local representations can be aggregated
   to make predictions at the whole image level.
1. As we proceed, deeper layers should be able to capture longer-range features of the 
   image, in a way similar to higher level vision in nature. 

Let's see how this translates into mathematics.


## Constraining the MLP

To start off, we can consider an MLP
with two-dimensional images $\mathbf{X}$ as inputs
and their immediate hidden representations
$\mathbf{H}$ similarly represented as matrices (they are two-dimensional tensors in code), where both $\mathbf{X}$ and $\mathbf{H}$ have the same shape.
Let that sink in.
We now imagine that not only the inputs but
also the hidden representations possess spatial structure.

Let $[\mathbf{X}]_{i, j}$ and $[\mathbf{H}]_{i, j}$ denote the pixel
at location $(i,j)$
in the input image and hidden representation, respectively.
Consequently, to have each of the hidden units
receive input from each of the input pixels,
we would switch from using weight matrices
(as we did previously in MLPs)
to representing our parameters
as fourth-order weight tensors $\mathsf{W}$.
Suppose that $\mathbf{U}$ contains biases,
we could formally express the fully connected layer as

$$\begin{aligned} \left[\mathbf{H}\right]_{i, j} &= [\mathbf{U}]_{i, j} + \sum_k \sum_l[\mathsf{W}]_{i, j, k, l}  [\mathbf{X}]_{k, l}\\ &=  [\mathbf{U}]_{i, j} +
\sum_a \sum_b [\mathsf{V}]_{i, j, a, b}  [\mathbf{X}]_{i+a, j+b}.\end{aligned}$$

The switch from $\mathsf{W}$ to $\mathsf{V}$ is entirely cosmetic for now
since there is a one-to-one correspondence
between coefficients in both fourth-order tensors.
We simply re-index the subscripts $(k, l)$
such that $k = i+a$ and $l = j+b$.
In other words, we set $[\mathsf{V}]_{i, j, a, b} = [\mathsf{W}]_{i, j, i+a, j+b}$.
The indices $a$ and $b$ run over both positive and negative offsets,
covering the entire image.
For any given location ($i$, $j$) in the hidden representation $[\mathbf{H}]_{i, j}$,
we compute its value by summing over pixels in $x$,
centered around $(i, j)$ and weighted by $[\mathsf{V}]_{i, j, a, b}$. Before we carry on, let's consider the total number of parameters required for a *single* layer in this parametrization: a $1000 \times 1000$ image (1 megapixel) is mapped to a $1000 \times 1000$ hidden representation. This requires $10^{12}$ parameters, far beyond what computers currently can handle.  

### Translation Invariance

Now let's invoke the first principle
established above: translation invariance :cite:`Zhang.ea.1988`.
This implies that a shift in the input $\mathbf{X}$
should simply lead to a shift in the hidden representation $\mathbf{H}$.
This is only possible if $\mathsf{V}$ and $\mathbf{U}$ do not actually depend on $(i, j)$. As such,
we have $[\mathsf{V}]_{i, j, a, b} = [\mathbf{V}]_{a, b}$ and $\mathbf{U}$ is a constant, say $u$.
As a result, we can simplify the definition for $\mathbf{H}$:

$$[\mathbf{H}]_{i, j} = u + \sum_a\sum_b [\mathbf{V}]_{a, b}  [\mathbf{X}]_{i+a, j+b}.$$


This is a *convolution*!
We are effectively weighting pixels at $(i+a, j+b)$
in the vicinity of location $(i, j)$ with coefficients $[\mathbf{V}]_{a, b}$
to obtain the value $[\mathbf{H}]_{i, j}$.
Note that $[\mathbf{V}]_{a, b}$ needs many fewer coefficients than $[\mathsf{V}]_{i, j, a, b}$ since it
no longer depends on the location within the image. Consequently, the number of parameters required is no longer $10^{12}$ but a much more reasonable $4 \times 10^6$: we still have the dependency on $a, b \in (-1000, 1000)$. In short, we have made significant progress. Time-delay neural networks (TDNNs) are some of the first examples to exploit this idea :cite:`Waibel.Hanazawa.Hinton.ea.1989`.

###  Locality

Now let's invoke the second principle: locality.
As motivated above, we believe that we should not have
to look very far away from location $(i, j)$
in order to glean relevant information
to assess what is going on at $[\mathbf{H}]_{i, j}$.
This means that outside some range $|a|> \Delta$ or $|b| > \Delta$,
we should set $[\mathbf{V}]_{a, b} = 0$.
Equivalently, we can rewrite $[\mathbf{H}]_{i, j}$ as

$$[\mathbf{H}]_{i, j} = u + \sum_{a = -\Delta}^{\Delta} \sum_{b = -\Delta}^{\Delta} [\mathbf{V}]_{a, b}  [\mathbf{X}]_{i+a, j+b}.$$
:eqlabel:`eq_conv-layer`

This reduces the number of parameters from $4 \times 10^6$ to $4 \Delta^2$, where $\Delta$ is typically smaller than $10$. As such, we reduced the number of parameters by another four orders of magnitude. Note that :eqref:`eq_conv-layer`, is what is called, in a nutshell, a *convolutional layer*. 
*Convolutional neural networks* (CNNs)
are a special family of neural networks that contain convolutional layers.
In the deep learning research community,
$\mathbf{V}$ is referred to as a *convolution kernel*,
a *filter*, or simply the layer's *weights* that are learnable parameters.

While previously, we might have required billions of parameters
to represent just a single layer in an image-processing network,
we now typically need just a few hundred, without
altering the dimensionality of either
the inputs or the hidden representations.
The price paid for this drastic reduction in parameters
is that our features are now translation invariant
and that our layer can only incorporate local information,
when determining the value of each hidden activation.
All learning depends on imposing inductive bias.
When that bias agrees with reality,
we get sample-efficient models
that generalize well to unseen data.
But of course, if those biases do not agree with reality,
e.g., if images turned out not to be translation invariant,
our models might struggle even to fit our training data.

This dramatic reduction in parameters brings us to our last desideratum, 
namely that deeper layers should represent larger and more complex aspects 
of an image. This can be achieved by interleaving nonlinearities and convolutional 
layers repeatedly. 

## Convolutions

Let's briefly review why :eqref:`eq_conv-layer` is called a convolution. 
In mathematics, the *convolution* between two functions :cite:`Rudin.1973`,
say $f, g: \mathbb{R}^d \to \mathbb{R}$ is defined as

$$(f * g)(\mathbf{x}) = \int f(\mathbf{z}) g(\mathbf{x}-\mathbf{z}) d\mathbf{z}.$$

That is, we measure the overlap between $f$ and $g$
when one function is "flipped" and shifted by $\mathbf{x}$.
Whenever we have discrete objects, the integral turns into a sum.
For instance, for vectors from
the set of square-summable infinite-dimensional vectors
with index running over $\mathbb{Z}$ we obtain the following definition:

$$(f * g)(i) = \sum_a f(a) g(i-a).$$

For two-dimensional tensors, we have a corresponding sum
with indices $(a, b)$ for $f$ and $(i-a, j-b)$ for $g$, respectively:

$$(f * g)(i, j) = \sum_a\sum_b f(a, b) g(i-a, j-b).$$
:eqlabel:`eq_2d-conv-discrete`

This looks similar to :eqref:`eq_conv-layer`, with one major difference.
Rather than using $(i+a, j+b)$, we are using the difference instead.
Note, though, that this distinction is mostly cosmetic
since we can always match the notation between
:eqref:`eq_conv-layer` and :eqref:`eq_2d-conv-discrete`.
Our original definition in :eqref:`eq_conv-layer` more properly
describes a *cross-correlation*.
We will come back to this in the following section.


## Channels
:label:`subsec_why-conv-channels`

Returning to our Waldo detector, let's see what this looks like.
The convolutional layer picks windows of a given size
and weighs intensities according to the filter $\mathsf{V}$, as demonstrated in :numref:`fig_waldo_mask`.
We might aim to learn a model so that
wherever the "waldoness" is highest,
we should find a peak in the hidden layer representations.

![Detect Waldo (image courtesy of William Murphy (Infomatique)).](../img/waldo-mask.jpg)
:width:`400px`
:label:`fig_waldo_mask`

There is just one problem with this approach.
So far, we blissfully ignored that images consist
of three channels: red, green, and blue. 
In sum, images are not two-dimensional objects
but rather third-order tensors,
characterized by a height, width, and channel,
e.g., with shape $1024 \times 1024 \times 3$ pixels. 
While the first two of these axes concern spatial relationships,
the third can be regarded as assigning
a multidimensional representation to each pixel location.
We thus index $\mathsf{X}$ as $[\mathsf{X}]_{i, j, k}$.
The convolutional filter has to adapt accordingly.
Instead of $[\mathbf{V}]_{a,b}$, we now have $[\mathsf{V}]_{a,b,c}$.

Moreover, just as our input consists of a third-order tensor,
it turns out to be a good idea to similarly formulate
our hidden representations as third-order tensors $\mathsf{H}$.
In other words, rather than just having a single hidden representation
corresponding to each spatial location,
we want an entire vector of hidden representations
corresponding to each spatial location.
We could think of the hidden representations as comprising
a number of two-dimensional grids stacked on top of each other.
As in the inputs, these are sometimes called *channels*.
They are also sometimes called *feature maps*,
as each provides a spatialized set
of learned features for the subsequent layer.
Intuitively, you might imagine that at lower layers that are closer to inputs,
some channels could become specialized to recognize edges while
others could recognize textures.

To support multiple channels in both inputs ($\mathsf{X}$) and hidden representations ($\mathsf{H}$),
we can add a fourth coordinate to $\mathsf{V}$: $[\mathsf{V}]_{a, b, c, d}$.
Putting everything together we have:

$$[\mathsf{H}]_{i,j,d} = \sum_{a = -\Delta}^{\Delta} \sum_{b = -\Delta}^{\Delta} \sum_c [\mathsf{V}]_{a, b, c, d} [\mathsf{X}]_{i+a, j+b, c},$$
:eqlabel:`eq_conv-layer-channels`

where $d$ indexes the output channels in the hidden representations $\mathsf{H}$. The subsequent convolutional layer will go on to take a third-order tensor, $\mathsf{H}$, as input.
We take
:eqref:`eq_conv-layer-channels`,
because of its generality, as
the definition of a convolutional layer for multiple channels, where $\mathsf{V}$ is a kernel or filter of the layer.

There are still many operations that we need to address.
For instance, we need to figure out how to combine all the hidden representations
to a single output, e.g., whether there is a Waldo *anywhere* in the image.
We also need to decide how to compute things efficiently,
how to combine multiple layers,
appropriate activation functions,
and how to make reasonable design choices
to yield networks that are effective in practice.
We turn to these issues in the remainder of the chapter.

## Summary and Discussion

In this section we derived the structure of convolutional neural networks from first principles. While it is unclear whether this was the route taken to the invention of CNNs, it is satisfying to know that they are the *right* choice when applying reasonable principles to how image processing and computer vision algorithms should operate, at least at lower levels. In particular, translation invariance in images implies that all patches of an image will be treated in the same manner. Locality means that only a small neighborhood of pixels will be used to compute the corresponding hidden representations. Some of the earliest references to CNNs are in the form of the Neocognitron :cite:`Fukushima.1982`. 

A second principle that we encountered in our reasoning is how to reduce the number of parameters in a function class without limiting its expressive power, at least, whenever certain assumptions on the model hold. We saw a dramatic reduction of complexity as a result of this restriction, turning computationally and statistically infeasible problems into tractable models. 

Adding channels allowed us to bring back some of the complexity that was lost due to the restrictions imposed on the convolutional kernel by locality and translation invariance. Note that it is quite natural to add channels other than just red, green, and blue. Many satellite 
images, in particular for agriculture and meteorology, have tens to hundreds of channels, 
generating hyperspectral images instead. They report data on many different wavelengths. In the following we will see how to use convolutions effectively to manipulate the dimensionality of the images they operate on, how to move from location-based to channel-based representations, and how to deal with large numbers of categories efficiently. 

## Exercises

1. Assume that the size of the convolution kernel is $\Delta = 0$.
   Show that in this case the convolution kernel
   implements an MLP independently for each set of channels. This leads to the Network in Network 
   architectures :cite:`Lin.Chen.Yan.2013`. 
1. Audio data is often represented as a one-dimensional sequence. 
    1. When might you want to impose locality and translation invariance for audio? 
    1. Derive the convolution operations for audio.
    1. Can you treat audio using the same tools as computer vision? Hint: use the spectrogram.
1. Why might translation invariance not be a good idea after all? Give an example. 
1. Do you think that convolutional layers might also be applicable for text data?
   Which problems might you encounter with language?
1. What happens with convolutions when an object is at the boundary of an image?
1. Prove that the convolution is symmetric, i.e., $f * g = g * f$.

[Discussions](https://discuss.d2l.ai/t/64)


1. Assume that the size of the convolution kernel is $\Delta = 0$.
   Show that in this case the convolution kernel
   implements an MLP independently for each set of channels. This leads to the Network in Network 
   architectures :cite:`Lin.Chen.Yan.2013`. 


## Answer to Exercise 1:

When the size of the convolution kernel is $\Delta = 0$, we're essentially using a $1 \times 1$ convolution. Let's examine what this means mathematically and intuitively.

Recall from the text that the general form of a convolutional layer with multiple channels is:

$$[{\mathsf{H}}]_{i,j,d} = \sum_{a = -\Delta}^{\Delta} \sum_{b = -\Delta}^{\Delta} \sum_c [{\mathsf{V}}]_{a, b, c, d} [{\mathsf{X}}]_{i+a, j+b, c}$$

When $\Delta = 0$, the ranges for $a$ and $b$ collapse to just $a=0$ and $b=0$, giving us:

$$[{\mathsf{H}}]_{i,j,d} = \sum_c [{\mathsf{V}}]_{0, 0, c, d} [{\mathsf{X}}]_{i, j, c}$$

This can be rewritten as:

$$[{\mathsf{H}}]_{i,j,d} = \sum_c w_{c,d} [{\mathsf{X}}]_{i, j, c}$$

where $w_{c,d} = [{\mathsf{V}}]_{0, 0, c, d}$ is the weight connecting input channel $c$ to output channel $d$.

**Intuitive explanation:**

This is exactly equivalent to applying a fully connected layer (i.e., an MLP layer) *independently* at each spatial location $(i,j)$ in the input. For each position $(i,j)$:

1. We take the vector of values across all input channels $[{\mathsf{X}}]_{i,j,:}$ (a vector of length = number of input channels)
2. We multiply it by the weight matrix $w$ (of size: input channels × output channels)
3. The result is the vector of values across all output channels $[{\mathsf{H}}]_{i,j,:}$ (a vector of length = number of output channels)

If we include biases (which were omitted in the derivation for simplicity), this becomes even more clearly an MLP:

$$[{\mathsf{H}}]_{i,j,d} = b_d + \sum_c w_{c,d} [{\mathsf{X}}]_{i, j, c}$$

where $b_d$ is the bias for output channel $d$.

**Why this is important:**

This $1 \times 1$ convolution (Network in Network) serves several purposes:
1. **Dimensionality reduction/expansion**: It can reduce or increase the number of channels without changing the spatial dimensions
2. **Nonlinear feature transformation**: When followed by a nonlinearity, it adds more complex transformations to the network
3. **Parameter efficiency**: It introduces nonlinearity across channels with minimal parameters compared to larger convolutions

The Network in Network architecture leverages this property to apply "micro neural networks" at each spatial location, enhancing the network's ability to learn complex functions while maintaining spatial structure and parameter efficiency.

2. Audio data is often represented as a one-dimensional sequence. 
    1. When might you want to impose locality and translation invariance for audio? 
    1. Derive the convolution operations for audio.
    1. Can you treat audio using the same tools as computer vision? Hint: use the spectrogram.


## Answer to Exercise 2:

### a. When might you want to impose locality and translation invariance for audio?

Locality and translation invariance are valuable for audio processing in several key scenarios:

1. **Pattern recognition**: Audio contains patterns like phonemes in speech, notes in music, or specific sounds (car horn, door slam) that should be recognized regardless of when they occur in the recording (translation invariance).

2. **Feature extraction**: Local spectral and temporal patterns often carry meaningful information. For example:
   - Attack-decay-sustain-release patterns in musical notes
   - Formants in speech sounds
   - Temporal dynamics in environmental sounds

3. **Noise reduction**: Local noise patterns can be identified and suppressed regardless of when they appear.

4. **Audio classification tasks**: Speaker identification, music genre classification, and environmental sound recognition all benefit from detecting specific patterns regardless of their precise timing.

5. **Acoustic event detection**: Identifying sounds like footsteps, glass breaking, or sirens benefits from translation invariance since the event's meaning doesn't depend on when it occurs.

### b. Derive the convolution operations for audio.

For a one-dimensional audio signal $x(t)$ where $t$ represents time, the convolution with a filter $v(t)$ is:

$$h(t) = (x * v)(t) = \sum_{a=-\Delta}^{\Delta} v(a) \cdot x(t+a)$$

Where:
- $h(t)$ is the output signal at time $t$
- $\Delta$ defines the receptive field of our filter (how many samples we look at)
- $v(a)$ is the filter/kernel weight at offset $a$

For multiple input and output channels (e.g., when processing features extracted from audio):

$$h(t, d) = \sum_{a=-\Delta}^{\Delta} \sum_c v(a, c, d) \cdot x(t+a, c)$$

Where:
- $c$ indexes the input channels
- $d$ indexes the output channels

This operation slides the filter $v$ across the audio signal $x$, computing a weighted sum at each position, capturing local patterns while maintaining translation invariance.

### c. Can you treat audio using the same tools as computer vision? Hint: use the spectrogram.

Yes, by transforming audio into visual representations, we can apply powerful computer vision techniques to audio analysis:

**Spectrograms** convert audio from a 1D time-domain signal into a 2D time-frequency representation by:
1. Applying Short-Time Fourier Transform (STFT) to segments of the audio
2. Computing the magnitude of the resulting complex values
3. Often applying a logarithmic scale (creating a log-spectrogram)

This transformation enables us to:
- Apply 2D convolutions to capture both time and frequency patterns simultaneously
- Detect patterns like formants in speech, harmonics in music, or specific acoustic signatures
- Leverage CNN architectures designed for images

**Additional audio-to-image representations:**
- **Mel spectrograms**: Spectrograms converted to the Mel scale to better match human auditory perception
- **MFCC** (Mel-Frequency Cepstral Coefficients): Compact features derived from Mel spectrograms
- **Constant-Q transforms**: Similar to spectrograms but with logarithmically spaced frequency bins
- **Wavelet scalograms**: Time-frequency representations using wavelet transforms

The advantage of this approach is that spectral-temporal patterns become visible as textures and shapes, allowing CNNs to identify complex audio patterns just as they identify visual patterns in images. Many state-of-the-art audio classification systems use this approach, treating audio analysis as an image recognition problem.

3. Why might translation invariance not be a good idea after all? Give an example. 


## Answer to Exercise 3:

While translation invariance is powerful for many computer vision tasks, there are several important scenarios where it can be problematic:

**1. Positional meaning in images**
As noted, the location of objects often carries critical semantic information. For example:
- A plane in the sky vs. on the runway represents fundamentally different situations (flying vs. parked)
- A car above a road likely indicates a dangerous accident rather than normal driving
- Medical imaging where anatomical location is crucial for diagnosis (a tumor's position relative to organs)

**2. Compositional relationships**
The relative positions of objects convey important relational information:
- In document analysis, the spatial arrangement of text and elements defines the document structure
- In scene understanding, "a person riding a bike" vs. "a bike next to a person" are distinguished primarily by spatial relationship
- In facial recognition, the relative positions of features (eyes above nose above mouth) are essential information

**3. Structural information**
Some tasks require understanding specific structures where position is intrinsic to meaning:
- In satellite imagery, the geography and relative positions are the primary information
- In architectural blueprints, the exact layout defines the design
- In analyzing charts and graphs, the position of points conveys the actual data values

**4. Position as a feature**
Sometimes position itself is the feature we care about:
- In pose estimation, the absolute positions of joints are what we're trying to predict
- In object localization tasks, we explicitly want to know where objects are located
- In image registration, precise spatial correspondence is the goal

**Addressing the limitation:**
Modern CNN architectures often incorporate positional information through techniques like:
- Position encodings
- Spatial attention mechanisms
- Region-based analysis (as in R-CNN variants)
- Specialized architectures for tasks where position matters (like U-Net for segmentation)

These approaches maintain the benefits of translation invariance where appropriate while preserving crucial positional information when needed.

4. Do you think that convolutional layers might also be applicable for text data?
   Which problems might you encounter with language?


## Answer to Exercise 4:

Yes, convolutional layers can be applied to text data and have been used successfully in certain NLP tasks, but with important limitations due to the fundamental differences between images and language.

### Applicability of Convolutions for Text

**How convolutions can work with text:**
- Text can be represented as a 1D sequence where each "pixel" is a word or character embedding
- 1D convolutions can slide over these sequences to detect local patterns
- Different filter sizes can capture n-grams of different lengths (e.g., 2-word, 3-word patterns)
- Hierarchical patterns can be learned through stacked convolutional layers

**Successful applications:**
- Text classification (sentiment analysis, topic identification)
- Named entity recognition
- Part-of-speech tagging
- Basic information extraction

### Challenges and Limitations

1. **Non-uniform importance of position:**
   As noted, language is not truly translation invariant. The meaning of words heavily depends on their position in grammatical structures. The word "not" completely changes meaning depending on position, and this effect is not uniform across the sequence.

2. **Long-range dependencies:**
   Language understanding often requires capturing dependencies between distant words. For example, in "The man who visited the store that had the sale which ended yesterday bought milk," understanding who bought milk requires spanning many words. Convolutions with practical filter sizes struggle with such dependencies.

3. **Hierarchical structure mismatch:**
   Language has a hierarchical structure (sentences, clauses, phrases) that doesn't neatly align with the fixed-size receptive fields of convolutions. Parse trees aren't easily captured by sliding windows.

4. **Variable-length inputs:**
   Text sequences vary dramatically in length, from single words to entire documents, creating challenges for fixed convolutional architectures.

5. **Word order vs. word proximity:**
   Convolutions treat proximity as the main indicator of relationship strength, which works for images but not always for text. In "John, despite not liking Mary, helped her," the relationship between "John" and "helped" is strong despite their distance.

6. **Context directionality:**
   In language, future words can be just as important as past words for understanding current words, but basic convolutions treat all directions within their receptive field equally.

### Modern Approaches

These limitations explain why more specialized architectures like recurrent neural networks (RNNs), and especially Transformers with their self-attention mechanism, have largely supplanted CNNs as the primary architecture for NLP tasks. However, CNNs remain valuable in specific NLP applications and as components in hybrid architectures.

5. What happens with convolutions when an object is at the boundary of an image?



## Answer to Exercise 5:

When an object is at the boundary of an image, convolutional operations face several challenges that can affect how the object is processed:

### The Boundary Problem

Without any special handling, convolutions at image boundaries have incomplete receptive fields (parts of the convolution kernel extend beyond the image). This leads to several issues:

1. **Information loss**: Features near boundaries receive reduced context because part of their neighborhood is missing, leading to less reliable feature extraction.

2. **Diminished representation**: As noted in your answer, objects at boundaries can appear "faded out" in feature maps because fewer pixels contribute to their representation.

3. **Inconsistent treatment**: An object at the image center has its full context considered, while the same object at the boundary has partial context, breaking translation invariance.

4. **Shrinking feature maps**: Each convolutional layer without padding reduces the spatial dimensions of feature maps, eventually "eroding" objects at boundaries.

### Common Solutions

Several techniques address boundary issues:

1. **Padding strategies**:
   - **Zero padding**: Fill boundary regions with zeros (most common)
   - **Reflection padding**: Mirror the image content at boundaries
   - **Replication padding**: Replicate the edge pixels outward
   - **Circular padding**: Wrap around to the opposite side (assumes periodicity)

2. **Valid vs. Same padding modes**:
   - **Valid padding** (no padding): Output size is smaller than input, boundary information is partially lost
   - **Same padding**: Output size matches input size, preserving spatial dimensions

3. **Dilated/atrous convolutions**: Expand the receptive field without increasing parameter count, helping capture more context even at boundaries

### Effects on Object Detection and Recognition

For objects at image boundaries:

1. **Detection challenges**: Objects partially outside the image or at the very edge are harder to detect reliably
   
2. **Context imbalance**: The network has asymmetric context around boundary objects, potentially affecting classification accuracy

3. **Practical implications**: This is why many object detection systems struggle with objects that are partially occluded by the image boundary

4. **Training considerations**: Data augmentation that places objects at different positions, including boundaries, helps networks learn to handle boundary cases more robustly

In practice, proper padding strategies and architectural designs can mitigate most boundary issues, but they remain an inherent limitation of the convolutional approach that practitioners should be aware of.

6. Prove that the convolution is symmetric, i.e., $f * g = g * f$.

## Answer to Exercise 6:

To prove that convolution is symmetric (commutative), we need to show that $f * g = g * f$ for any two functions $f$ and $g$ where the convolution is defined.

Let's start with the definition of convolution given in the text for continuous functions:

$$(f * g)(\mathbf{x}) = \int f(\mathbf{z}) g(\mathbf{x}-\mathbf{z}) d\mathbf{z}$$

I'll prove the symmetry property step by step:

**Step 1:** Let's compute $(g * f)(\mathbf{x})$ using the definition:

$$(g * f)(\mathbf{x}) = \int g(\mathbf{z}) f(\mathbf{x}-\mathbf{z}) d\mathbf{z}$$

**Step 2:** Introduce a change of variables. Let $\mathbf{w} = \mathbf{x} - \mathbf{z}$, which means $\mathbf{z} = \mathbf{x} - \mathbf{w}$. When we make this substitution, we also need to adjust the differential element:

$$d\mathbf{z} = -d\mathbf{w}$$

The negative sign appears because we're changing the direction of the variable.

**Step 3:** Substituting these into our expression:

$$(g * f)(\mathbf{x}) = \int g(\mathbf{x} - \mathbf{w}) f(\mathbf{w}) (-d\mathbf{w})$$
$$(g * f)(\mathbf{x}) = -\int g(\mathbf{x} - \mathbf{w}) f(\mathbf{w}) d\mathbf{w}$$

**Step 4:** Pull out the negative sign and rearrange:

$$(g * f)(\mathbf{x}) = \int f(\mathbf{w}) g(\mathbf{x} - \mathbf{w}) d\mathbf{w}$$

**Step 5:** Recognize that this is exactly the expression for $(f * g)(\mathbf{x})$:

$$(g * f)(\mathbf{x}) = \int f(\mathbf{w}) g(\mathbf{x} - \mathbf{w}) d\mathbf{w} = (f * g)(\mathbf{x})$$

Therefore, $f * g = g * f$, proving that convolution is symmetric (commutative).

**For the discrete case:**
The proof follows similarly. If we have:

$$(f * g)(i) = \sum_a f(a) g(i-a)$$

Then:

$$(g * f)(i) = \sum_a g(a) f(i-a)$$

Using the substitution $b = i-a$ (which means $a = i-b$), we get:

$$(g * f)(i) = \sum_b g(i-b) f(b) = \sum_b f(b) g(i-b) = (f * g)(i)$$

This symmetry property is important because it means the order of functions in a convolution doesn't matter—we get the same result either way. In the context of neural networks, this mathematical property isn't directly exploited, since we typically use cross-correlation (which is not symmetric) rather than true convolution, as mentioned in the text.